# 🎯 [WEL03_M10] 가상 수질검사 데이터 위험 신호 대시보드 시스템 통합 패키지 완성
### Google Colab 실습 환경 (장병 복지·지원 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/WEL03/wel03_m10_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장병 복지·지원 (Level 2) |
| **미션 ID** | WEL03_M10 |
| **미션 주제** | 가상 수질검사 데이터 위험 신호 대시보드 시스템 통합 패키지 완성 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
전장에서는 급수차, 정수 장비, 우물, 하천 취수원 등 급수원의 수질 이상을 빨리 알아채지 못하면 장병의 건강과 부대 활동에 바로 영향을 주지만, 센서 값은 잡음과 고장이 많아 오경보로 불필요하게 급수를 끊는 일도 생긴다. 러시아-우크라이나 전쟁 사례에서 학습 초점으로 삼은 전장 위생·건강 위험 관리를 반영해, 본 프로젝트는 가상 수질검사 데이터를 분석해 위험 신호를 데이터로 판단하는 과제를 다룬다. 교육용 가상 급수원 8곳의 pH, 탁도, 잔류염소, 중금속, 대장균 측정 자료를 코드로 만들어 스키마 정의, 센서 결측·이상치 정제, 기준치 Baseline 판정, 복합 오염 징후 분류 모델, 신뢰도 지수와 오경보 감축, 상황판 대시보드, 급수 중단 HITL 결재선, 대량 오염 부하 시험, AAR 임계치 재조정, 통합 패키지까지 단계별로 구현한다. 최종 목표는 수질 위험 신호를 근거와 함께 빠르게 제시하되, 실제 음용 판정과 급수 중단은 공식 기준과 의무(예방의학) 담당 확인, 지휘관 결심에 따르도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/wel03_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 결재 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/wel03_data_spec.md`, `docs/wel03_aar_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/wel03_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/wel03_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/wel03_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 10. 가상 수질검사 데이터 위험 신호 대시보드 시스템 통합 패키지 완성
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

가상 수질검사 데이터 위험 신호 대시보드 시스템 통합 패키지를 완성한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **가상 수질검사 데이터 위험 신호 대시보드를 하나의 통합 패키지로 완성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **입력 검증, 전처리, 재조정 기준치 판정, 랜덤 포레스트 위험 분류, 신뢰도 등급, 사람 검토 큐 등록, 상황판 요약을 한 파일의 파이프라인으로 묶는다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/main_wel03.py, README.md, docs/wel03_final_report.pdf(md)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 실무 즉시 도입 가능성, 운용 신뢰도 및 전략적 기대효과

#### 미션 PASS 조건

- 정량 기준: End-to-End 통합 파이프라인 실행 성공률 100%, 테스트 커버리지 85% 이상


- 진행 기준: 원클릭 실행 스크립트 동작 확인 및 최종 발표 산출물 작성 완료


- 검증 방법: 최종 검증 위원회(평가관) 모의 시연 및 전체 기능 통합 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""입력 검증, 전처리, 재조정 기준치 판정과 랜덤 포레스트 위험 분류, 신뢰도 등급, 인간 검토 큐 등록, 상황판 요약까지 가상 수질검사 위험 신호 대시보드를 한 파일로 통합 실행하고, 여러 시나리오 End-to-End 실행·자체 점검(함수 커버리지)·지휘관 브리핑 요약을 출력한다."""
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------------------------
# [교육용가상자료] 야전 급수원 가상 수질 센서 데이터 생성기 (실제 부대·위치·검사 결과 없음)
# 가상 급수원 8곳 x 14일 x 1시간 간격. 실제 수질(true_*)·센서 측정값·오염 사건·센서 고장 정답을 함께 만든다.
# 기준치는 교육용 가정값이며, 실제 음용 판정은 공인 검사 결과와 의무(예방의학) 담당 기준에 따른다.
# ---------------------------------------------------------------------------
PARAMS = ["ph", "turbidity", "chlorine", "metal", "ecoli"]
SOURCES = [("W01", "water_truck", 1.0, 2.0), ("W02", "water_truck", 3.5, 1.2), ("W03", "purification_unit", 2.2, 4.1),
           ("W04", "purification_unit", 5.0, 3.3), ("W05", "well", 0.8, 5.6), ("W06", "well", 4.4, 6.0),
           ("W07", "river_intake", 6.3, 1.8), ("W08", "river_intake", 6.9, 4.9)]
BASE = {"water_truck": (7.2, 0.25, 0.75, 1.5), "purification_unit": (7.0, 0.20, 0.85, 1.2),
        "well": (6.8, 0.45, 0.50, 3.0), "river_intake": (7.3, 0.55, 0.60, 2.0)}   # pH, 탁도(NTU), 잔류염소(mg/L), 중금속(ug/L)
LIMITS = {"ph": (5.8, 8.5), "turbidity": (1.0, 2.0), "chlorine": (0.2, 0.1), "metal": (5.0, 10.0), "ecoli": (1, 1)}  # (주의, 위험) 가정 기준
EVENT_TYPES = {  # 오염 사건: 변수별 최대 변화량, 최대치 도달 시간(h)
    "sewage": ({"turbidity": 2.8, "chlorine": -0.5, "ecoli": 30, "ph": -0.3}, 4),
    "metal": ({"metal": 22.0, "ph": -0.5}, 4),
    "disinfect_fail": ({"chlorine": -0.8, "ecoli": 6}, 6),
    "chemical": ({"ph": 1.7, "turbidity": 1.0}, 3),
    "creeping_metal": ({"metal": 13.0}, 18),
}
FAULT_TYPES = ["stuck", "spike", "drift", "dropout"]
NOISE = {"ph": 0.04, "turbidity": 0.06, "chlorine": 0.03, "metal": 0.35}
DIGITS = {"ph": 2, "turbidity": 2, "chlorine": 2, "metal": 1}


def make_dataset(days=14, seed=7, n_events=6, n_faults=10, events=None, faults=None, sources=None):
    """events: [(급수원 번호, 사건 유형, 시작 시각 h, 지속 h, 강도)], faults: [(급수원 번호, 고장 유형, 항목, 시작 h, 지속 h)]"""
    rng = np.random.default_rng(seed)
    sources = sources or SOURCES
    T = days * 24
    tt = np.arange(T)
    hour = tt % 24
    night = (hour < 6) | (hour >= 20)
    rain = np.zeros(T)
    for _ in range(max(1, days // 4)):
        s0, d = int(rng.integers(0, T - 14)), int(rng.integers(6, 14))
        rain[s0:s0 + d] = np.maximum(rain[s0:s0 + d], rng.uniform(0.6, 1.0))
    if events is None:
        events = [(int(rng.integers(len(sources))), str(rng.choice(list(EVENT_TYPES))), int(rng.integers(min(24, T // 4), max(T // 4 + 1, T - 48))),
                   int(rng.integers(10, 30)), 1.0) for _ in range(n_events)]
    if faults is None:
        faults = [(int(rng.integers(len(sources))), str(rng.choice(FAULT_TYPES)), str(rng.choice(["ph", "turbidity", "chlorine", "metal"])),
                   int(rng.integers(12, max(13, T - 30))), int(rng.integers(6, 24))) for _ in range(n_faults)]
    out = []
    for si, (sid, stype, x, y) in enumerate(sources):
        ph0, tb0, cl0, mt0 = BASE[stype]
        tr = {"ph": ph0 + 0.08 * np.sin(2 * np.pi * hour / 24) + rng.normal(0, 0.03, T),
              "turbidity": tb0 * (1 + 0.1 * rng.normal(0, 1, T)),
              "chlorine": (cl0 * np.exp(-0.022 * hour) if stype == "water_truck" else cl0 + rng.normal(0, 0.03, T)),
              "metal": mt0 + rng.normal(0, 0.2, T), "ecoli": np.zeros(T)}
        if stype in ("river_intake", "well"):
            tr["turbidity"] = tr["turbidity"] + rain * (0.9 if stype == "river_intake" else 0.45)
        f_all, etype = np.zeros(T), np.array(["none"] * T, dtype=object)
        for (s_i, et, onset, dur, scale) in events:
            if s_i != si:
                continue
            delta, ramp = EVENT_TYPES[et]
            f = np.clip((tt - onset + 1) / ramp, 0, 1) * ((tt >= onset) & (tt < onset + dur + ramp))
            for k, v in delta.items():
                tr[k] = tr[k] + v * f * scale
            etype[f > 0] = et
            f_all = np.maximum(f_all, f)
        tr["chlorine"] = np.clip(tr["chlorine"], 0.02, None)
        tr["turbidity"] = np.clip(tr["turbidity"], 0.05, None)
        tr["metal"] = np.clip(tr["metal"], 0.1, None)
        grade = np.where(f_all >= 0.2, 2, 0)
        mild = (tr["turbidity"] > LIMITS["turbidity"][0]) | (tr["chlorine"] < LIMITS["chlorine"][0]) | (tr["metal"] > LIMITS["metal"][0])
        grade = np.where((grade == 0) & mild, 1, grade)
        ms = {}
        for p in ["ph", "turbidity", "chlorine", "metal"]:
            sd = NOISE[p] * np.where(night, 1.5, 1.0) * (tr[p] if p == "turbidity" else 1)
            ms[p] = np.round(np.clip(tr[p] + rng.normal(0, 1, T) * sd, 0, None), DIGITS[p])
        ms["ecoli"] = (rng.poisson(tr["ecoli"]) + (rng.random(T) < 0.004)).astype(float)
        fault, fparam = np.array(["none"] * T, dtype=object), np.array(["-"] * T, dtype=object)
        for (s_i, ft, p, s0, d) in faults:
            if s_i != si:
                continue
            seg = slice(s0, min(T, s0 + d))
            if ft == "stuck":
                ms[p][seg] = ms[p][s0]
            elif ft == "spike":
                seg = slice(s0, s0 + 1)
                ms[p][s0] = {"ph": 11.8, "turbidity": 38.0, "chlorine": 0.0, "metal": 75.0}[p]
            elif ft == "drift":
                ramp = np.linspace(0, 1, seg.stop - s0)
                ms[p][seg] = np.clip(ms[p][seg] + ramp * {"ph": -1.6, "turbidity": 2.6, "chlorine": -0.75, "metal": 12.0}[p], 0, None).round(DIGITS[p])
            elif ft == "dropout":
                for q in PARAMS:
                    ms[q][seg] = np.nan
                p = "all"
            fault[seg], fparam[seg] = ft, p
        df = pd.DataFrame({"source_id": sid, "source_type": stype, "x_km": x, "y_km": y, "t": tt,
                           "cond": np.where(rain > 0, "storm", np.where(night, "night", "day")),
                           **{p: ms[p] for p in PARAMS}, **{f"true_{p}": tr[p] for p in PARAMS},
                           "event": etype, "fault": fault, "fault_param": fparam, "grade": grade})
        out.append(df)
    return pd.concat(out, ignore_index=True)
# ---------------------------------------------------------------------------

# ---- 전처리 공통 함수: 물리 범위 점검 -> 고정값(stuck) 표시 -> 단발 스파이크 제거 -> 결측 보정 ----
PHYS = {"ph": (0, 14), "turbidity": (0, 1000), "chlorine": (0, 10), "metal": (0, 1000), "ecoli": (0, 100000)}
SPIKE_K = {"day": 6.0, "night": 7.0, "storm": 9.0}          # 상황별(주간/야간/악천후) 스파이크 판정 배수
MIN_JUMP = {"ph": 0.8, "turbidity": 1.5, "chlorine": 0.3, "metal": 6.0}
STUCK_RUN, SHORT_GAP = 6, 3


def clean(df, spike_k=None, min_jump=None):
    """급수원별 시계열을 정제하고 항목별 처리 내역(<항목>_flag: range/stuck/spike/interp/long_gap)을 남긴다."""
    spike_k = spike_k or SPIKE_K
    min_jump = MIN_JUMP if min_jump is None else min_jump
    out = df.sort_values(["source_id", "t"]).reset_index(drop=True)
    for sid, idx in out.groupby("source_id").groups.items():
        idx = np.asarray(idx)
        kk = out.loc[idx, "cond"].map(spike_k).to_numpy(float)
        for p in PARAMS:
            v = out.loc[idx, p].to_numpy(float).copy()
            fl = np.array([""] * len(v), dtype=object)
            lo, hi = PHYS[p]
            bad = (v < lo) | (v > hi)
            v[bad], fl[bad] = np.nan, "range"
            if p != "ecoli":
                chg = np.r_[True, v[1:] != v[:-1]]                       # 같은 값이 STUCK_RUN시간 이상 반복 = 고정값 고장 의심
                rid = np.cumsum(chg)
                rlen = np.bincount(rid)[rid]
                fl[(rlen >= STUCK_RUN) & ~chg & ~np.isnan(v)] = "stuck"
                d = np.abs(np.diff(v[~np.isnan(v)]))
                sig = 1.4826 * np.median(d) + 1e-6 if len(d) else 1.0
                up, dn = v - np.r_[np.nan, v[:-1]], v - np.r_[v[1:], np.nan]
                thr = np.maximum(kk * sig, min_jump[p])
                spike = (np.abs(up) > thr) & (np.abs(dn) > thr) & (np.sign(up) == np.sign(dn))   # 튀었다가 바로 돌아온 값만
                v[spike], fl[spike] = np.nan, "spike"
            s = pd.Series(v)
            na = s.isna().to_numpy()
            grp = np.cumsum(np.r_[True, na[1:] != na[:-1]])
            glen = pd.Series(na.astype(int)).groupby(grp).transform("sum").to_numpy()
            lin, ff = s.interpolate(limit_area="inside").to_numpy(), s.ffill().bfill().to_numpy()
            short, longg = na & (glen <= SHORT_GAP), na & (glen > SHORT_GAP)
            v = np.where(short, np.where(np.isnan(lin), ff, lin), np.where(longg, ff, v))
            fl[short & (fl == "")] = "interp"
            fl[longg] = "long_gap"
            out.loc[idx, p] = v
            out.loc[idx, f"{p}_flag"] = fl
    return out

# ---- 가정 기준치 판정(Baseline)과 시계열 특징 공통 함수 ----
GRADE_NAME = {0: "안전", 1: "주의", 2: "위험"}


def rule_grade(df):
    """기준치 1차 필터: 위험(음용 부적합 의심) = 한 항목이라도 위험 기준 초과, 주의 = 주의 기준 초과."""
    ph, tb, cl, mt, ec = (df[p].to_numpy(float) for p in PARAMS)
    danger = (ph < LIMITS["ph"][0]) | (ph > LIMITS["ph"][1]) | (tb > LIMITS["turbidity"][1]) | (cl < LIMITS["chlorine"][1]) \
        | (mt > LIMITS["metal"][1]) | (ec >= LIMITS["ecoli"][1])
    caution = (tb > LIMITS["turbidity"][0]) | (cl < LIMITS["chlorine"][0]) | (mt > LIMITS["metal"][0])
    return np.where(danger, 2, np.where(caution, 1, 0))


def reasons(row):
    r = []
    if not LIMITS["ph"][0] <= row["ph"] <= LIMITS["ph"][1]:
        r.append(f"pH {row['ph']:.2f}")
    if row["turbidity"] > LIMITS["turbidity"][0]:
        r.append(f"탁도 {row['turbidity']:.2f}NTU")
    if row["chlorine"] < LIMITS["chlorine"][0]:
        r.append(f"잔류염소 {row['chlorine']:.2f}mg/L")
    if row["metal"] > LIMITS["metal"][0]:
        r.append(f"중금속 {row['metal']:.1f}ug/L")
    if row["ecoli"] >= 1:
        r.append(f"대장균 {row['ecoli']:.0f}")
    return ", ".join(r) or "-"


FEATS = [f"{p}{s}" for p in ["ph", "turbidity", "chlorine", "metal"] for s in ["", "_d1", "_d3", "_dev", "_sd6"]] + \
        ["ecoli", "ecoli_3h", "storm", "night", "n_flag", "truck", "purif", "well", "river"]


def add_features(c):
    """시간 순서를 지키는(과거 값만 쓰는) 급수원별 특징: 변화량, 24시간 기준선 대비 편차, 6시간 변동성, 대장균 누적 등."""
    c = c.sort_values(["source_id", "t"]).reset_index(drop=True)
    g = c.groupby("source_id")
    for p in ["ph", "turbidity", "chlorine", "metal"]:
        c[f"{p}_d1"] = g[p].diff().fillna(0)
        c[f"{p}_d3"] = g[p].diff(3).fillna(0)
        base = g[p].transform(lambda s: s.shift(1).rolling(24, min_periods=4).median())
        c[f"{p}_dev"] = (c[p] - base).fillna(0)
        c[f"{p}_sd6"] = g[p].transform(lambda s: s.rolling(6, min_periods=2).std()).fillna(0)
    c["ecoli_3h"] = g["ecoli"].transform(lambda s: s.rolling(3, min_periods=1).sum())
    c["storm"], c["night"] = (c["cond"] == "storm").astype(int), (c["cond"] == "night").astype(int)
    flags = [f"{p}_flag" for p in PARAMS if f"{p}_flag" in c]
    c["n_flag"] = sum(c[f].isin(["stuck", "spike", "long_gap", "range"]).astype(int) for f in flags) if flags else 0
    for k, name in [("truck", "water_truck"), ("purif", "purification_unit"), ("well", "well"), ("river", "river_intake")]:
        c[k] = (c["source_type"] == name).astype(int)
    return c

CONFIG = {"ecoli_3h_min": 2, "metal_dev": 1.0, "metal_min": 5.0,            # 미션 9 재조정 값
          "conf_high": 60, "scale": {"ph": 0.35, "turbidity": 0.6, "chlorine": 0.2, "metal": 3.0}}
README = {"1. 목적": "가상 수질 센서 자료로 급수원 위험 신호를 조기에 알리는 판단 보조 도구",
          "2. 실행 방법": "python main_wel03.py (원클릭 실행, 외부 파일·네트워크 불필요)",
          "3. 입력 형식": "급수원 ID·유형·시각·pH·탁도·잔류염소·중금속·대장균(미션 1 스키마)",
          "4. 판정 규칙": "가정 기준치 + 재조정 규칙 + 모델, 신뢰도 High만 급수 중단 검토 상신",
          "5. 인간 검토": "급수 중단·재개는 예방의학 담당 검토 후 지휘관 승인, 자동 확정 없음",
          "6. 한계와 주의": "교육용 가상 자료, 실제 판정은 공식 수질 기준과 의무·환경 담당 확인",
          "7. 연동 확장": "인접 부대 급수 정보 교환용 표준 필드(JSON) API를 차기 과제로 둔다"}


def validate(df):
    ok = df["source_id"].astype(str).str.match(r"^W\d{2}$") & df["t"].ge(0)
    return df[ok].copy(), int((~ok).sum())


def tuned_grade(c):
    ph, tb, cl, mt, ec = (c[p].to_numpy(float) for p in PARAMS)
    eco = (c["ecoli_3h"].to_numpy() >= CONFIG["ecoli_3h_min"]) | ((ec >= 1) & ((tb > LIMITS["turbidity"][0]) | (cl < LIMITS["chlorine"][0])))
    trend = (c["metal_dev"].to_numpy() > CONFIG["metal_dev"]) & (mt > CONFIG["metal_min"])
    danger = (ph < LIMITS["ph"][0]) | (ph > LIMITS["ph"][1]) | (tb > LIMITS["turbidity"][1]) | (cl < LIMITS["chlorine"][1]) \
        | (mt > LIMITS["metal"][1]) | eco | trend
    caution = (tb > LIMITS["turbidity"][0]) | (cl < LIMITS["chlorine"][0]) | (mt > LIMITS["metal"][0]) | (ec >= 1)
    return np.where(danger, 2, np.where(caution, 1, 0))


def confidence(c, alert):
    S = CONFIG["scale"]
    n_dev = sum(((c[f"{p}_dev"].abs() > S[p]) | (c[f"{p}_d3"].abs() > S[p])).astype(int) for p in S) + (c["ecoli_3h"] >= 2)
    a = pd.Series(alert.astype(int), index=c.index)
    run = a.groupby([c["source_id"], (a != a.groupby(c["source_id"]).shift()).cumsum()]).cumsum()
    g = c.groupby("source_id")
    flagged = g["n_flag"].transform(lambda s: s.rolling(6, min_periods=1).max()) > 0
    # 드리프트 의심: 한 항목만 6시간 중 5시간 이상 같은 방향으로 꾸준히 변함(잡음이 커지는 야간에도 유지되는 기준)
    mono = pd.concat([g[f"{p}_d1"].transform(lambda s: np.sign(s).rolling(6, min_periods=3).sum()).abs() >= 4 for p in S], axis=1).any(axis=1)
    drift = (n_dev <= 1) & mono & (c["ecoli_3h"] < 2)
    return 40 * np.minimum(n_dev, 3) / 3 + 25 * np.minimum(run, 3) / 3 + 20 * (~flagged) + 15 * (~drift)


def train_model():
    tr = pd.concat([add_features(clean(make_dataset(seed=s, n_events=14, n_faults=14))) for s in (7, 8)], ignore_index=True)
    return RandomForestClassifier(n_estimators=150, max_depth=10, class_weight="balanced_subsample", random_state=0, n_jobs=1).fit(tr[FEATS], tr["grade"])


def run_pipeline(raw, model):
    data, n_bad = validate(raw)
    c = add_features(clean(data))
    c["rule"] = tuned_grade(c)
    c["ml"] = model.predict(c[FEATS])
    c["final"] = np.maximum(c["rule"], c["ml"])                          # 둘 중 하나라도 위험이면 위험(놓침 방지)
    alert = (c["final"] == 2).to_numpy()
    c["conf"] = np.where(alert, confidence(c, alert), np.nan)
    queue = c[alert & (c["conf"] >= CONFIG["conf_high"])].groupby("source_id").agg(first=("t", "min"), hours=("t", "size"))
    queue["status"] = "PENDING(지휘관 결정 대기)"
    return c, queue, n_bad


def self_tests(model):
    res = {}
    clean_df = make_dataset(days=2, seed=50, events=[], faults=[])
    c, q, _ = run_pipeline(clean_df, model)
    res["오염 없는 자료에서 급수 중단 상신 0건"] = len(q) == 0
    ev = make_dataset(days=2, seed=50, events=[(0, "sewage", 20, 12, 1.0)], faults=[])
    c, q, _ = run_pipeline(ev, model)
    res["하수 유입 급수원(W01) 상신"] = "W01" in q.index
    res["상신 건은 모두 PENDING(자동 확정 없음)"] = q["status"].str.startswith("PENDING").all()
    bad = ev.copy(); bad.loc[3, "source_id"] = "??"
    res["손상 레코드 격리"] = run_pipeline(bad, model)[2] == 1
    dr = make_dataset(days=2, seed=50, events=[], faults=[(1, "drift", "turbidity", 10, 18)])
    c, q, _ = run_pipeline(dr, model)
    n_alert = int((c.loc[c["source_id"] == "W02", "final"] == 2).sum())     # 드리프트 구간 위험 판정 시간 중 High 상신 시간이 절반 미만이면 억제 작동
    res["센서 드리프트 상신 시간 절반 미만으로 억제"] = ("W02" not in q.index) or q.loc["W02", "hours"] < 0.5 * n_alert
    res["기준치 필터·판정 근거 문구 생성"] = len(reasons(c[c["source_id"] == "W02"].iloc[40])) > 0 and rule_grade(c).shape[0] == len(c)
    res["결측 보정 후 남은 결측 0"] = int(c[PARAMS].isna().sum().sum()) == 0
    res["운영 문서 7개 장 작성"] = all(len(v) > 10 for v in README.values())
    return res


# ---- 자체 점검(함수 호출 추적으로 커버리지 측정) ----
t_all = time.perf_counter()
called = set()
own = {n for n, f in list(globals().items()) if callable(f) and getattr(f, "__module__", "") == "__main__" and hasattr(f, "__code__")}
sys.setprofile(lambda fr, ev, arg: called.add(fr.f_code.co_name) if ev == "call" else None)
model = train_model()
tests = self_tests(model)
e2e = []
for s in (11, 21, 31, 41, 51):
    try:
        c, q, nb = run_pipeline(make_dataset(seed=s, n_events=10, n_faults=14), model)
        e2e.append((s, True, (c["final"] == c["grade"]).mean(), len(q)))
    except Exception as e:
        e2e.append((s, False, np.nan, 0))
sys.setprofile(None)
cov = len(own & called) / len(own)
print(f"[교육용가상자료] 통합 실행 {time.perf_counter() - t_all:.1f}초 (모델 학습 포함)")
print("[End-to-End 시나리오 5종]")
for s, ok, acc, nq in e2e:
    print(f"  seed {s}: {'성공' if ok else '실패'}, 최종 판정 정확도 {acc:.1%}, 급수 중단 검토 상신 {nq}곳")
rate = np.mean([x[1] for x in e2e])
print(f"E2E 성공률 {rate:.0%} (목표 100%: {'달성' if rate == 1 else '미달'})")
print("[자체 점검]")
for k, v in tests.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print(f"자체 점검 {sum(tests.values())}/{len(tests)} 통과, 함수 커버리지 {cov:.0%}({len(own & called)}/{len(own)}, 목표 85% 이상: {'달성' if cov >= 0.85 else '미달'})")
print(f"운영 문서(README) {len(README)}개 장: " + ", ".join(README))

# ---- 브리핑 요약(벤치마크 seed 11, 운용 시점 기준) ----
c, q, _ = run_pipeline(make_dataset(seed=11, n_events=10, n_faults=14), model)
d = c["grade"] == 2
print("\n[지휘관 브리핑 요약]")
print(f"1. 급수원 8곳 14일 모의: 위험 재현율 {((c['final'] == 2) & d).sum() / d.sum():.1%}, 종합 정확도 {(c['final'] == c['grade']).mean():.1%}")
print(f"2. 급수 중단 검토 상신 {len(q)}곳({', '.join(q.index)}): 모두 예방의학 담당 검토·지휘관 결정 대기")
print("3. 센서 이상(드리프트·스파이크·단절)은 재측정·교정 요청으로 분리해 불필요한 단수를 줄인다")
print("4. 실제 음용 판정은 공식 수질 기준과 의무(예방의학)·환경 담당 확인에 따른다(본 도구는 판단 보조)")

last = c[c["t"] == c["t"].max()].set_index("source_id")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(last.index, last["final"], color=[{0: "tab:green", 1: "gold", 2: "red"}[g] for g in last["final"]])
ax[0].set_yticks([0, 1, 2], ["safe", "caution", "danger"]); ax[0].set_title("Final status by source (last hour)")
ax[1].bar([str(x[0]) for x in e2e], [x[2] for x in e2e], color="tab:blue"); ax[1].set_ylim(0.9, 1.0)
ax[1].set_title("E2E accuracy by scenario seed"); ax[1].set_xlabel("seed")
plt.suptitle("Integrated water risk dashboard package (synthetic)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 통합 실행 6.6초 (모델 학습 포함)
[End-to-End 시나리오 5종]
  seed 11: 성공, 최종 판정 정확도 98.6%, 급수 중단 검토 상신 6곳
  seed 21: 성공, 최종 판정 정확도 99.1%, 급수 중단 검토 상신 6곳
  seed 31: 성공, 최종 판정 정확도 97.0%, 급수 중단 검토 상신 6곳
  seed 41: 성공, 최종 판정 정확도 97.7%, 급수 중단 검토 상신 7곳
  seed 51: 성공, 최종 판정 정확도 98.0%, 급수 중단 검토 상신 7곳
E2E 성공률 100% (목표 100%: 달성)
[자체 점검]
  [PASS] 오염 없는 자료에서 급수 중단 상신 0건
  [PASS] 하수 유입 급수원(W01) 상신
  [PASS] 상신 건은 모두 PENDING(자동 확정 없음)
  [PASS] 손상 레코드 격리
  [PASS] 센서 드리프트 상신 시간 절반 미만으로 억제
  [PASS] 기준치 필터·판정 근거 문구 생성
  [PASS] 결측 보정 후 남은 결측 0
  [PASS] 운영 문서 7개 장 작성
자체 점검 8/8 통과, 함수 커버리지 100%(11/11, 목표 85% 이상: 달성)
운영 문서(README) 7개 장: 1. 목적, 2. 실행 방법, 3. 입력 형식, 4. 판정 규칙, 5. 인간 검토, 6. 한계와 주의, 7. 연동 확장

[지휘관 브리핑 요약]
1. 급수원 8곳 14일 모의: 위험 재현율 98.4%, 종합 정확도 98.6%
2. 급수 중단 검토 상신 6곳(W02, W03, W04, W05, W07, W08): 모두 예방의학 담당 검토·지휘관 결정 대기
3. 센서 이상(드리프트·스파이크·단절)은 재측정·교정 요청으로 분리해 불필요한 단수를 줄인다
4. 실제 음용 판정은 공식 수질 기준과 의무(예방의학)·환경 담당 확인에 따른다(본 도구는 판단 보조)
```
</details>
